# Modelo preliminar de clasificación

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

RANDOM_STATE = 42
DATA = Path("../data/train.csv")
df = pd.read_csv(DATA)
df.shape

(7613, 5)

## 1. Limpieza mínima del texto

In [2]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
NON_ALPHA_RE = re.compile(r"[^a-z\s]")

def limpiar(texto):
    texto = texto.lower()
    texto = URL_RE.sub(" ", texto)
    texto = MENTION_RE.sub(" ", texto)
    texto = texto.replace("#", " ")
    texto = NON_ALPHA_RE.sub(" ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df["text_clean"] = df["text"].apply(limpiar)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
df[["text", "text_clean"]].head()

,text,text_clean
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada
2,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...
3,"13,000 people receive #wildfires evacuation or...",people receive wildfires evacuation orders in ...
4,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...


## 2. Split train/test

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"], df["target"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=df["target"]
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")

Train: 6002  Test: 1501


## 3. Vectorización: TF-IDF con unigramas y bigramas

In [4]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    min_df=2,
    stop_words="english"
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
X_train_vec.shape

(6002, 8000)

## 4. Modelos candidatos

Se prueban dos algoritmos simples y estándar para clasificación de texto como línea base:

- **Regresión Logística**: modelo lineal, fácil de interpretar (los coeficientes indican qué palabras/bigramas empujan hacia cada clase), buen punto de partida para texto disperso tipo TF-IDF.
- **Naive Bayes multinomial**: clásico para clasificación de texto, asume independencia entre términos, muy rápido y suele funcionar bien como baseline en este tipo de problemas.

In [5]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Multinomial Naive Bayes": MultinomialNB(),
}

resultados = []
predicciones = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train_vec, y_train)
    pred = modelo.predict(X_test_vec)
    predicciones[nombre] = pred
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    })

resultados_df = pd.DataFrame(resultados).sort_values("f1", ascending=False)
resultados_df

,modelo,accuracy,precision,recall,f1
0,Logistic Regression,0.81479,0.853516,0.682813,0.758681
1,Multinomial Naive Bayes,0.81479,0.872428,0.662500,0.753108


## 5. Resultados preliminares

Ambos modelos muestran mayor precision que recall, siendo conservadores al clasificar un tweet como desastre real. La Regresión Logística, seleccionada como modelo preliminar por tener el mejor F1 (0.759), tiene recall de 0.68 frente a 0.66 de Naive Bayes.

In [6]:
mejor_modelo_nombre = resultados_df.iloc[0]["modelo"]
cm = confusion_matrix(y_test, predicciones[mejor_modelo_nombre])
pd.DataFrame(cm, index=["Real: 0", "Real: 1"], columns=["Pred: 0", "Pred: 1"])

,Pred: 0,Pred: 1
Real: 0,786,75
Real: 1,203,437
